# 03_silver_quarantine_handler

## Purpose
Handle quarantined (malformed) records from Silver transformations:
- View quarantine table contents
- Analyze failure reasons
- Provide reprocessing capability
- Generate quarantine reports

In [0]:
from pyspark.sql import functions as F

CAT = "zillow"
QUARANTINE = "zillow_quarantine"

# Ensure schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CAT}.{QUARANTINE}")

In [0]:
# View quarantine table summary
print("="*60)
print("QUARANTINE SUMMARY")
print("="*60)

quarantine_table = f"{CAT}.{QUARANTINE}.failed_records"

try:
    df_quarantine = spark.table(quarantine_table)
    total_count = df_quarantine.count()
    
    print(f"\nTotal quarantined records: {total_count:,}")
    
    # Summary by source table and failure reason
    print("\nBreakdown by Source Table:")
    display(
        df_quarantine
        .groupBy("source_table")
        .agg(
            F.count("*").alias("record_count"),
            F.min("failed_dt").alias("earliest_failure"),
            F.max("failed_dt").alias("latest_failure")
        )
        .orderBy(F.desc("record_count"))
    )
    
    print("\nBreakdown by Failure Reason:")
    display(
        df_quarantine
        .groupBy("failure_reason")
        .count()
        .orderBy(F.desc("count"))
    )
    
except Exception as e:
    print(f"No quarantine records found or table doesn't exist: {e}")
    print("This is normal if no records have failed validation.")

In [0]:
# View sample quarantined records
print("\n" + "="*60)
print("SAMPLE QUARANTINED RECORDS")
print("="*60)

try:
    display(
        spark.sql(f"""
            SELECT 
                source_table,
                failure_reason,
                failed_dt,
                SUBSTRING(record_json, 1, 200) as record_preview
            FROM {quarantine_table}
            ORDER BY failed_dt DESC
            LIMIT 20
        """)
    )
except:
    print("No quarantine records to display.")

In [0]:
def reprocess_quarantine_records(source_table_filter: str = None):
    """
    Attempt to reprocess quarantined records.
    
    This function:
    1. Reads records from quarantine
    2. Parses the JSON record
    3. Applies fixes (if possible)
    4. Attempts re-validation
    5. Moves successfully fixed records back to processing queue
    
    Args:
        source_table_filter: Optional filter for specific source table
    """
    print("Reprocessing quarantine records...")
    
    try:
        df = spark.table(quarantine_table)
        
        if source_table_filter:
            df = df.filter(F.col("source_table") == source_table_filter)
        
        count = df.count()
        print(f"Found {count:,} records to evaluate")
        
        if count == 0:
            return
        
        # Parse JSON and check if we can fix
        # For now, just report - actual fix logic would be custom per failure type
        df_parsed = df.withColumn("record", F.from_json("record_json", "map<string,string>"))
        
        # Check which records might be fixable
        # Example: Records with just empty string issues
        df_fixable = df_parsed.filter(
            F.col("failure_reason").contains("Missing required field")
        )
        
        fixable_count = df_fixable.count()
        print(f"Potentially fixable records: {fixable_count:,}")
        
        # In a real scenario, you would:
        # 1. Apply fixes to the records
        # 2. Re-validate
        # 3. Insert fixed records back to Bronze/Silver
        # 4. Remove from quarantine
        
        print("\nNote: Manual review required for these records.")
        print("Automatic reprocessing not implemented - custom fix logic needed.")
        
    except Exception as e:
        print(f"Error during reprocessing: {e}")

# Uncomment to run reprocessing
# reprocess_quarantine_records()

In [0]:
def cleanup_old_quarantine_records(days_to_keep: int = 30):
    """
    Remove quarantine records older than specified days.
    
    Args:
        days_to_keep: Number of days to retain records
    """
    print(f"Cleaning up quarantine records older than {days_to_keep} days...")
    
    try:
        cutoff_date = F.date_sub(F.current_date(), days_to_keep)
        
        # Count records to be deleted
        delete_count = spark.sql(f"""
            SELECT COUNT(*) as cnt 
            FROM {quarantine_table} 
            WHERE DATE(failed_dt) < DATE_SUB(CURRENT_DATE(), {days_to_keep})
        """).collect()[0]["cnt"]
        
        print(f"Records to delete: {delete_count:,}")
        
        if delete_count > 0:
            # Use Delta DELETE
            spark.sql(f"""
                DELETE FROM {quarantine_table} 
                WHERE DATE(failed_dt) < DATE_SUB(CURRENT_DATE(), {days_to_keep})
            """)
            print(f"Deleted {delete_count:,} old quarantine records")
        else:
            print("No old records to clean up")
            
    except Exception as e:
        print(f"Error during cleanup: {e}")

# Uncomment to run cleanup (keeps last 30 days)
# cleanup_old_quarantine_records(30)

In [0]:
# Generate quarantine report
print("\n" + "="*60)
print("QUARANTINE REPORT")
print("="*60)

try:
    report_df = spark.sql(f"""
        SELECT 
            source_table,
            failure_reason,
            DATE(failed_dt) as failure_date,
            COUNT(*) as record_count
        FROM {quarantine_table}
        GROUP BY source_table, failure_reason, DATE(failed_dt)
        ORDER BY failure_date DESC, source_table, record_count DESC
    """)
    
    display(report_df)
    
    # Save report as a view
    report_view = f"{CAT}.{QUARANTINE}.v_quarantine_report"
    spark.sql(f"CREATE OR REPLACE VIEW {report_view} AS {report_df._jdf.queryExecution().simpleString()}")
    print(f"\nReport view created: {report_view}")
    
except Exception as e:
    print(f"Could not generate report: {e}")